# LLM-Assisted Analog Amplifier Sizing

**An LLM-guided analog amplifier sizing flow using gm/ID methodology, ngspice SPICE simulation, and iterative root-cause diagnosis on the SKY130 open-source PDK.**

---

## Tool Versions

| Tool | Version | Purpose |
|------|---------|---------|  
| ngspice | 46 | SPICE simulation |
| SKY130 PDK | Open-source | 130nm process (bundled in repo) |
| Python | 3.11 | Agent logic, gm/ID LUT queries |
| Claude Code | Latest | LLM-driven interactive design agent (VS Code extension) |
| FastAPI | 0.118+ | CircuitCollector simulation server |
| Conda/Miniforge | Latest | Environment management |

## License

MIT

---

---

## Section 1: Environment Setup

This section installs all prerequisites on a fresh Ubuntu/Debian system:
1. System build dependencies
2. ngspice 46 (built from source)
3. Miniforge (conda)
4. Clone the project repository
5. Create two separate conda environments

> **Note:** If you already have any of these installed, you can skip the corresponding cells.

### 1.1 Install System Build Dependencies

In [ ]:
%%bash
sudo apt-get update -qq
sudo apt-get install -y -qq \
    build-essential autoconf automake libtool bison flex \
    libx11-dev libxaw7-dev libreadline-dev libfftw3-dev \
    wget curl git \
    2>&1 | tail -3
echo "System dependencies installed."

### 1.2 Build and Install ngspice 46

ngspice is the open-source SPICE simulator used for all circuit simulations. We build version 46 from source and install it to `$HOME/ngspice46/`.

In [ ]:
%%bash
set -e

NGSPICE_VERSION=46
INSTALL_DIR="$HOME/ngspice${NGSPICE_VERSION}"

echo "=== Downloading ngspice-${NGSPICE_VERSION} source ==="
cd /tmp
wget -q "https://sourceforge.net/projects/ngspice/files/ng-spice-rework/${NGSPICE_VERSION}/ngspice-${NGSPICE_VERSION}.tar.gz/download" \
     -O ngspice-${NGSPICE_VERSION}.tar.gz
tar xzf ngspice-${NGSPICE_VERSION}.tar.gz
cd ngspice-${NGSPICE_VERSION}

echo "=== Configuring ==="
mkdir -p release && cd release
../configure \
    --prefix="${INSTALL_DIR}" \
    --with-x \
    --with-readline=yes \
    --with-fftw3=yes \
    --enable-xspice \
    --enable-cider \
    --enable-openmp \
    --enable-klu \
    > /dev/null 2>&1

echo "=== Compiling (this may take a few minutes) ==="
make -j$(nproc) > /dev/null 2>&1

echo "=== Installing to ${INSTALL_DIR} ==="
make install > /dev/null 2>&1

# Verify
${INSTALL_DIR}/bin/ngspice --version 2>&1 | head -5
echo ""
echo "ngspice installed at ${INSTALL_DIR}/bin/ngspice"

In [ ]:
import os, shutil

NGSPICE_BIN_DIR = os.path.expanduser("~/ngspice46/bin")

# Add ngspice to PATH for this notebook session and all child processes
os.environ["PATH"] = NGSPICE_BIN_DIR + ":" + os.environ["PATH"]

ngspice_path = shutil.which("ngspice")
assert ngspice_path is not None, f"ngspice not found! Expected at {NGSPICE_BIN_DIR}/ngspice"
print(f"ngspice ready: {ngspice_path}")

In [ ]:
import os

# Write ~/.spiceinit for SKY130 PDK
# Only num_threads is active; all compatibility/perf options are commented out
spiceinit_path = os.path.expanduser("~/.spiceinit")
spiceinit_content = """\
* .spiceinit for use with Skywater PDK and ngspice
* set ngbehavior=hsa   ; HSA mode disabled
* set skywaterpdk      ; skip some checks for faster lib loading
* set ng_nomodcheck    ; don't check the model parameters
set num_threads=8      ; CPU processor cores available
* option noinit        ; don't print operating point data
* option klu           ; select KLU as matrix solver
"""

with open(spiceinit_path, "w") as f:
    f.write(spiceinit_content)

print(f"Written {spiceinit_path}")
print(open(spiceinit_path).read())

### 1.3 Install Miniforge (Conda)

Two separate conda environments are needed — one for the simulation server, one for the agent.

In [ ]:
%%bash
set -e

# Check if conda is available (command -v, or check common install paths)
if command -v conda &> /dev/null; then
    echo "Conda already installed: $(conda --version)"
    exit 0
fi

for DIR in "$HOME/miniforge3" "$HOME/anaconda3" "$HOME/miniconda3"; do
    if [ -x "${DIR}/bin/conda" ]; then
        echo "Conda already installed at ${DIR}"
        ${DIR}/bin/conda --version
        exit 0
    fi
done

echo "=== Installing Miniforge ==="
wget -q "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh" \
     -O /tmp/miniforge.sh
bash /tmp/miniforge.sh -b -p "$HOME/miniforge3"
rm /tmp/miniforge.sh

echo "Miniforge installed to $HOME/miniforge3"

In [ ]:
import os, shutil, subprocess

# Find conda executable
conda_path = shutil.which("conda")
if not conda_path:
    # Check common locations
    for candidate in [
        os.path.expanduser("~/miniforge3/bin/conda"),
        os.path.expanduser("~/anaconda3/bin/conda"),
        os.path.expanduser("~/miniconda3/bin/conda"),
        "/usr/bin/conda",
    ]:
        if os.path.isfile(candidate):
            conda_path = candidate
            os.environ["PATH"] = os.path.dirname(candidate) + ":" + os.environ["PATH"]
            break

assert conda_path, "conda not found! Please install Miniforge and restart the notebook."

# Store for later use
CONDA = conda_path
print(f"conda ready: {CONDA}")
r = subprocess.run([CONDA, "--version"], capture_output=True, text=True)
print(r.stdout.strip())

### 1.4 Clone the Project Repository

The repository contains **AnalogAgent** and **CircuitCollector** as parallel subfolders, along with the SKY130 PDK and pre-generated gm/ID look-up tables.

In [ ]:
%%bash
set -e

REPO_URL="https://github.com/jiyuanduan001-oss/LLM_Amplifier_Sizing.git"
REPO_DIR="$HOME/LLM_Amplifier_Sizing"

if [ -d "${REPO_DIR}" ]; then
    echo "Repository already cloned at ${REPO_DIR}"
else
    echo "=== Cloning repository ==="
    git clone "${REPO_URL}" "${REPO_DIR}"
fi

echo ""
echo "Repository structure:"
echo "  ${REPO_DIR}/"
echo "  ├── AnalogAgent/       (LLM agent + gm/ID LUT + skill stack)"
echo "  └── CircuitCollector/  (simulation server + ngspice runner + SKY130 PDK)"

In [ ]:
import os

REPO_DIR = os.path.expanduser("~/LLM_Amplifier_Sizing")
ANALOG_AGENT_DIR = os.path.join(REPO_DIR, "AnalogAgent")
CIRCUIT_COLLECTOR_DIR = os.path.join(REPO_DIR, "CircuitCollector")

assert os.path.isdir(ANALOG_AGENT_DIR), f"AnalogAgent not found at {ANALOG_AGENT_DIR}"
assert os.path.isdir(CIRCUIT_COLLECTOR_DIR), f"CircuitCollector not found at {CIRCUIT_COLLECTOR_DIR}"

print(f"AnalogAgent:      {ANALOG_AGENT_DIR}")
print(f"CircuitCollector: {CIRCUIT_COLLECTOR_DIR}")

### 1.5 Create Conda Environments

Two **separate** conda environments are required:

| Environment | Purpose | Key Packages |
|-------------|---------|------|
| `CircuitCollector` | Runs the FastAPI simulation server, calls ngspice | FastAPI, uvicorn, jinja2, scipy |
| `Agent` | Runs AnalogAgent tools (LUT queries, simulation bridge, plotting) | numpy, matplotlib, httpx, pandas, scipy |

> **Important:** Using a single environment will NOT work. The CircuitCollector server must run in its own environment.

In [ ]:
import subprocess, os

REPO_DIR = os.path.expanduser("~/LLM_Amplifier_Sizing")

# Required pip packages for CircuitCollector
CC_PIP_PACKAGES = [
    "fastapi==0.118.0", "uvicorn==0.35.0", "httpx>=0.28",
    "jinja2>=3.1", "toml>=0.10", "numpy>=2.0", "pandas>=2.0",
    "scipy>=1.14", "redis>=5.0", "python-dotenv>=1.0",
    "pydantic>=2.0", "pydantic-settings>=2.0", "sqlalchemy>=2.0",
    "requests>=2.31", "matplotlib>=3.8",
]

# Check if CircuitCollector env already exists
r = subprocess.run([CONDA, "env", "list"], capture_output=True, text=True)
if "CircuitCollector" in r.stdout:
    print("CircuitCollector environment already exists.")
else:
    # Try creating from environment.yml first
    env_file = os.path.join(REPO_DIR, "CircuitCollector", "CircuitCollector", "environment.yml")
    print("Creating CircuitCollector environment (this may take several minutes)...")
    r = subprocess.run(
        [CONDA, "env", "create", "-f", env_file],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        # Fallback: create a clean Python 3.11 environment
        print("environment.yml failed (platform mismatch). Falling back to clean env...")
        subprocess.run(
            [CONDA, "create", "-n", "CircuitCollector", "python=3.11", "pip", "-y"],
            capture_output=True, text=True
        )
    print("Base environment created.")

# Unconditionally install all required pip packages as a safety net
print("Installing required pip packages...")
r = subprocess.run(
    [CONDA, "run", "-n", "CircuitCollector",
     "pip", "install", "--quiet"] + CC_PIP_PACKAGES,
    capture_output=True, text=True
)
if r.returncode != 0:
    print(f"pip install warning:\n{r.stderr[-300:]}")
else:
    print("Pip packages installed.")

# Install CircuitCollector as editable package
print("Installing CircuitCollector package...")
r = subprocess.run(
    [CONDA, "run", "-n", "CircuitCollector",
     "pip", "install", "-e", os.path.join(REPO_DIR, "CircuitCollector"), "--quiet"],
    capture_output=True, text=True
)
print("Done." if r.returncode == 0 else f"Error: {r.stderr[-300:]}")

In [ ]:
import subprocess, os

REPO_DIR = os.path.expanduser("~/LLM_Amplifier_Sizing")

# Required pip packages for Agent
AGENT_PIP_PACKAGES = [
    "numpy>=2.0", "matplotlib>=3.8", "requests>=2.31", "toml>=0.10",
    "jinja2>=3.1", "pandas>=2.0", "httpx>=0.28", "scipy>=1.14",
]

# Check if Agent env already exists
r = subprocess.run([CONDA, "env", "list"], capture_output=True, text=True)
if "Agent" in r.stdout:
    print("Agent environment already exists.")
else:
    print("Creating Agent environment...")
    subprocess.run(
        [CONDA, "create", "-y", "-n", "Agent", "python=3.11", "pip"],
        check=True, capture_output=True,
    )
    subprocess.run(
        [CONDA, "run", "-n", "Agent", "pip", "install", "--quiet"] + AGENT_PIP_PACKAGES,
        check=True, capture_output=True,
    )
    # Install CircuitCollector as editable package (absolute path)
    subprocess.run(
        [CONDA, "run", "-n", "Agent", "pip", "install", "-e",
         os.path.join(REPO_DIR, "CircuitCollector"), "--quiet"],
        check=True, capture_output=True,
    )
    print("Agent environment created.")

# Verify both environments
r = subprocess.run([CONDA, "env", "list"], capture_output=True, text=True)
for line in r.stdout.strip().split("\n"):
    if "CircuitCollector" in line or "Agent" in line:
        print(f"  {line.strip()}")


---

## Section 2: Start and Verify CircuitCollector Server

CircuitCollector is a FastAPI server that handles:
- Dynamic circuit registration (converting SPICE netlists to parameterized templates)
- Testbench generation using Jinja2 templates
- ngspice simulation execution
- Result parsing (AC, DC, noise, slew rate, output swing, operating point data)

The server runs in the **CircuitCollector** conda environment with ngspice on the PATH.

In [ ]:
import subprocess, os, time

REPO_DIR = os.path.expanduser("~/LLM_Amplifier_Sizing")
CIRCUIT_COLLECTOR_DIR = os.path.join(REPO_DIR, "CircuitCollector")
NGSPICE_BIN_DIR = os.path.expanduser("~/ngspice46/bin")
SERVER_LOG = "/tmp/circuitcollector.log"
SERVER_PID_FILE = "/tmp/circuitcollector.pid"

# Stop any existing CircuitCollector server so we start fresh
subprocess.run(
    "pkill -f 'CircuitCollector.api.main:app' 2>/dev/null; sleep 1; true",
    shell=True,
)

# Start the CircuitCollector server fully DETACHED from this notebook kernel.
#
# Why detached:
#   The LLM agent (launched later in a separate terminal / IDE) calls
#   simulate_circuit() via HTTP. If the server is a child of this notebook,
#   closing the notebook or restarting the kernel kills the server.
#
#   nohup  -> ignore SIGHUP when the parent shell exits
#   setsid -> detach from the controlling terminal (new session)
#   < /dev/null, >log, 2>&1 -> close stdin, redirect stdout/stderr
#   &, disown -> don't block, remove from shell's job table
#
# Note: bash -c uses single quotes so $PATH is expanded by bash at run time.
# $PATH is wrapped in double quotes because PATH inherited from Jupyter may
# contain directories with spaces (e.g. "/mnt/d/Microsoft VS Code/bin" on WSL).
cmd = (
    f"nohup setsid bash -c '"
    f'export PATH="{NGSPICE_BIN_DIR}:$PATH" && '
    f"cd {CIRCUIT_COLLECTOR_DIR} && "
    f"exec {CONDA} run --no-capture-output -n CircuitCollector "
    f"uvicorn CircuitCollector.api.main:app --host 0.0.0.0 --port 8001"
    f"' > {SERVER_LOG} 2>&1 < /dev/null & "
    f"echo $! > {SERVER_PID_FILE}; disown"
)
subprocess.run(cmd, shell=True, executable="/bin/bash")
time.sleep(5)

with open(SERVER_PID_FILE) as f:
    server_pid = int(f.read().strip())

print(f"CircuitCollector server started (PID: {server_pid}, detached)")
print(f"  Log:          {SERVER_LOG}")
print(f"  PID file:     {SERVER_PID_FILE}")
print(f"  ngspice PATH: {NGSPICE_BIN_DIR}")
print("")
print("Server will keep running after this notebook closes.")
print("To stop it manually later:  kill $(cat /tmp/circuitcollector.pid)")

In [ ]:
import urllib.request, json, time

SERVER_URL = "http://localhost:8001"
max_retries = 10

for i in range(max_retries):
    try:
        r = urllib.request.urlopen(f"{SERVER_URL}/health", timeout=5)
        data = json.loads(r.read())
        print(f"CircuitCollector server is running: {data}")
        break
    except Exception:
        print(f"  Waiting... ({i+1}/{max_retries})")
        time.sleep(3)
else:
    # Server didn't come up — surface the log to help debugging
    log_path = "/tmp/circuitcollector.log"
    if os.path.exists(log_path):
        with open(log_path) as f:
            tail = "".join(f.readlines()[-30:])
        print(f"Last 30 lines of {log_path}:\n{tail}")
    raise RuntimeError(
        "CircuitCollector server failed to start. "
        "Check /tmp/circuitcollector.log for details."
    )

### Quick Simulation Test

Verify the full pipeline works: send a test parameter set to CircuitCollector and confirm we get back simulation results from ngspice.

In [ ]:
import urllib.request, json

SERVER_URL = "http://localhost:8001"

# Test with a 5T OTA parameter set
payload = json.dumps({
    "params": {
        "M1_L": 0.5,  "M1_WL_ratio": 5.0,  "M1_M": 1,
        "M2_L": 0.5,  "M2_WL_ratio": 5.0,  "M2_M": 1,
        "M3_L": 0.5,  "M3_WL_ratio": 5.0,  "M3_M": 1,
        "M4_L": 0.5,  "M4_WL_ratio": 5.0,  "M4_M": 1,
        "M5_L": 1.0,  "M5_WL_ratio": 5.0,  "M5_M": 1,
        "M6_L": 1.0,  "M6_WL_ratio": 5.0,  "M6_M": 1,
        "ibias": 5e-6,
        # quick-test: skip slow measurements (mismatch MC ~22s, noise/slew/swing ~few s each).
        # The full sizing flow re-enables them per-call when needed.
        "measure_mismatch": False,
        "measure_noise": False,
        "measure_slew_rate": False,
        "measure_output_swing": False,
    },
    "base_config_path": "config/skywater/opamp/5tota_single.toml",
    "spec_list": ["AC", "DC", "GBW_PM"],
}).encode()

print("Sending test simulation to CircuitCollector...")
req = urllib.request.Request(
    f"{SERVER_URL}/simulate/",
    data=payload,
    headers={"Content-Type": "application/json"},
)
r = urllib.request.urlopen(req, timeout=120)
result = json.loads(r.read())

if "specs" in result:
    print("\nSimulation successful!\n")
    specs = result["specs"]
    print("Key results:")
    for key, unit, scale in [
        ("dcgain_", "dB", 1),
        ("gain_bandwidth_product_", "MHz", 1e-6),
        ("phase_margin", "deg", 1),
        ("power", "uW", 1e6),
    ]:
        val = specs.get(key)
        if val is not None:
            print(f"  {key}: {val * scale:.2f} {unit}")
    print("\nCircuitCollector <-> ngspice pipeline is working correctly.")
else:
    print("Unexpected result:")
    print(json.dumps(result, indent=2)[:500])

---

## Section 3: Run the Sizing Agent

The AnalogAgent skill stack (`skills/analog-amplifier/`) is a collection of plain markdown files that any LLM agent can read. The Python tools (`tools/`, `scripts/`) and CircuitCollector HTTP API are also agent-agnostic. Any LLM agent that can read files and execute Python/bash commands can run the full sizing flow.

The examples below show setup for **Claude Code** (VS Code extension) and **Codex**, but the same approach works with any tool-calling LLM.

### 3.1 Install an LLM Agent (Example: VS Code + Claude Code)

In [ ]:
%%bash
set -e

# Install VS Code if not present
if command -v code &> /dev/null; then
    echo "VS Code already installed: $(code --version 2>/dev/null | head -1)"
else
    echo "=== Installing VS Code ==="
    wget -q "https://go.microsoft.com/fwlink/?LinkID=760868" -O /tmp/vscode.deb
    sudo dpkg -i /tmp/vscode.deb 2>/dev/null || sudo apt-get install -f -y -qq 2>&1 | tail -3
    rm /tmp/vscode.deb
    echo "VS Code installed."
fi

# Install Claude Code extension
echo ""
echo "=== Installing Claude Code VS Code extension ==="
code --install-extension anthropic.claude-code 2>&1 || echo "(If this fails, install it manually from the VS Code Extensions marketplace)"
echo "Done."

### 3.2 Launch the Sizing Flow

After all environments are prepared, follow these steps.

> **Important:** The CircuitCollector server started in Section 2 is detached and
> keeps running in the background, so the LLM agent can reach it over HTTP even
> after this notebook is closed. If you ever need to stop it:
> ```bash
> kill $(cat /tmp/circuitcollector.pid)
> ```
> If you ever need to restart it manually (e.g., after a reboot):
> ```bash
> conda activate CircuitCollector
> export PATH="$HOME/ngspice46/bin:$PATH"
> cd ~/LLM_Amplifier_Sizing/CircuitCollector
> nohup setsid uvicorn CircuitCollector.api.main:app --host 0.0.0.0 --port 8001 \
>     > /tmp/circuitcollector.log 2>&1 < /dev/null &
> echo $! > /tmp/circuitcollector.pid
> disown
> ```

**Step 1: Activate the Agent environment and open your IDE**

```bash
conda activate Agent
cd ~/LLM_Amplifier_Sizing/AnalogAgent
code .       # or any IDE / terminal your LLM agent runs in
```

**Step 2: Point the LLM agent to the skill stack**

The design skills live in `skills/analog-amplifier/` as plain markdown files. Any LLM agent can read them — there is no IDE-specific discovery mechanism. The entry point is `skills/analog-amplifier/SKILL.md`, which describes the full design flow and references all sub-skills.

For **Claude Code**: paste the sizing prompt below and reference the skills directory.
For **Codex** or other agents: paste the same prompt — the agent will read the skill stack and follow the design flow.

### Example Prompts

**Step 1: Set your specifications**

Open `spec-form-template.md` in the repo root and fill in your target specs (VDD, CL, DC gain, GBW, phase margin, slew rate, noise, etc.). This template is the single source of truth for the specs the agent will size against.

**Step 2: Choose a netlist**

Reference netlists are provided in `AnalogAgent/examples/` for every supported topology — 5T OTA (single / cascode / wide-swing cascode load) and Two-Stage Miller (single / cascode / wide-swing cascode load). For example, `examples/5tota_variants/5tota_single.sp` is the plain 5-transistor OTA.

> **Note:** You can also supply your own netlist. Just drop a `.sp` SPICE subcircuit into the working directory (or anywhere the LLM agent can read it) and reference it in your prompt. The agent will parse the topology, register it with CircuitCollector, and size it with the same flow.

**Step 3: Prompt the LLM agent**

The skill stack is in `skills/analog-amplifier/`. Any LLM agent that can read files and execute Python/bash can follow the design flow.

**Claude Code:**

```
Use the skills to size the 5tota_single; use the specs in the template.
```

**Codex / other agents:**

```
Use the skills to size the 5tota_single; use the specs in the template.
```

---

> ### Bash Command Permissions
>
> During execution, the LLM agent will run **many bash commands** to call the ngspice simulator, read/write netlist and config files, query the gm/ID LUT, and so on.
>
> **You have two options:**
>
> 1. **Manual approval (default)** — A dialog will pop up for every command. You must click **Yes / Allow** each time for the flow to continue. Leaving the session unattended will stall the agent.
>
> 2. **Auto-approve mode (recommended for end-to-end runs)** — Enable auto-approve / bypass mode in your agent's IDE so commands run without confirmation. This lets the sizing flow run unattended from start to finish.

---

The agent will automatically:
1. Load specs from `spec-form-template.md` and the chosen netlist
2. Identify the topology (e.g., 5T OTA with single mirror load)
3. Register it with CircuitCollector
4. Size all devices using gm/ID methodology + SKY130 LUT data
5. Run SPICE simulation via ngspice
6. Diagnose and fix any spec failures
7. Optionally run CMA-ES numerical optimization
8. Produce a full design review with spec comparison table